In [1]:
import sys
sys.path.append("..")

from src.io_utils import load_csv_to_dataframe
from src.schemas import PathConfig

import pandas as pd

path_config = PathConfig()

path_config.ensure_directories()


In [ ]:
df_credit = load_csv_to_dataframe(file_path=f"{path_config.input_dir}/creditcard.csv")

df_credit

In [ ]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from imblearn.combine import SMOTEENN
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline as PipelineImblearn
from imblearn.over_sampling import SMOTE

In [ ]:
df_credit_copy = df_credit.copy()

In [ ]:
X = df_credit_copy.drop('Class', axis=1)
y = df_credit_copy['Class']

# Split (helt opåverkad rådata)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Definiera kolumntransformering
preprocessor = ColumnTransformer(
    transformers=[
        ('scaler', RobustScaler(), ['Amount', 'Time']) # Skalar BARA Amount och Time
    ],
    remainder='passthrough' # Släpper igenom V1...V28 exakt som de är!
)

In [ ]:
# Sätt ihop i en Pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1))
])

pipeline2 = PipelineImblearn(steps=[
    ("preprocessor", preprocessor),
    ("smote", SMOTE(sampling_strategy= 0.1, random_state=42)),
    ("model", RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1))
])

smote = SMOTE(sampling_strategy= 0.1, random_state=42)
smote_smoteen = SMOTEENN(smote=smote, random_state=42)

pipeline3 = PipelineImblearn(steps=[
    ("preprocessor", preprocessor),
    ("hybrid_sampler", smote_smoteen),
    ("model", RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1))
])

pipeline4 = PipelineImblearn(steps=[
    ("preprocessor", preprocessor),
    ("hybrid_sampler", SMOTEENN(random_state=42)),
    ("model", RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1))
])

In [ ]:
model_path_no_smote = path_config.model_dir / "credit_fraud_pipeline_untouched_dataset.joblib"

#Träna hela kedjan med en enda fit()-sats
if model_path_no_smote.exists():
    # Om filen finns -> ladda den
    pipe_no_smote = joblib.load(model_path_no_smote)

#Spara HELA pipelinen (scalern + modellen) till en enda fil
else:
    # Om filen inte finns -> träna modellen och spara den
    pipe_no_smote = pipeline.fit(X_train, y_train)
    joblib.dump(pipe_no_smote, model_path_no_smote)

In [ ]:
model_path_smote = path_config.model_dir / "credit_fraud_pipeline_smote.joblib"

if model_path_smote.exists():
    pipe_smote = joblib.load(model_path_smote)

else:
    pipe_smote = pipeline2.fit(X_train, y_train)
    joblib.dump(pipe_smote, model_path_smote)

In [ ]:
model_path_smoteenn = path_config.model_dir / "credit_fraud_pipeline_smoteenn.joblib"

if model_path_smoteenn.exists():
    pipe_smoteenn = joblib.load(model_path_smoteenn)

else:
    pipe_smoteenn = pipeline4.fit(X_train, y_train)
    joblib.dump(pipe_smoteenn, model_path_smoteenn)



In [ ]:

model_path_smoteenn_smote = path_config.model_dir / "credit_fraud_pipeline_smoteenn_smote.joblib"

if model_path_smoteenn_smote.exists():
    pipe_smoteenn_smote = joblib.load(model_path_smoteenn_smote)

else:
    pipe_smoteenn_smote = pipeline3.fit(X_train, y_train)
    joblib.dump(pipe_smoteenn_smote, model_path_smoteenn_smote)



In [ ]:
# Utvärdera på testdata (Pipelinen kör .transform() automatiskt på X_test)
y_probs_no_smote = pipe_no_smote.predict_proba(X_test)[:, 1]

In [ ]:
y_probs_smote = pipe_smote.predict_proba(X_test)[:, 1]

In [ ]:
y_probs_smoteenn = pipe_smoteenn.predict_proba(X_test)[:, 1]

In [ ]:
y_probs_smoteenn_smote = pipe_smoteenn_smote.predict_proba(X_test)[:, 1]

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, auc

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_probs_no_smote)
pr_auc = auc(recall, precision)

print(f"Area Under Precision-Recall Curve with full untouched dataset: {pr_auc:.4f}")

plt.figure(figsize=(7, 5))
plt.plot(recall, precision, label=f"PR-AUC = {pr_auc:.3f}")
plt.xlabel('Recall (How many frauds we caught)')
plt.ylabel('Precision (How many flags were actual fraud)')
plt.title('Precision-Recall Curve with full untouched dataset')
plt.legend()
plt.tight_layout()
plt.grid(True)
plt.show()

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_probs_smote)
pr_auc = auc(recall, precision)

print(f"Area Under Precision-Recall Curve  with SMOTE: {pr_auc:.4f}")

plt.figure(figsize=(7, 5))
plt.plot(recall, precision, label=f"PR-AUC = {pr_auc:.3f}")
plt.xlabel('Recall (How many frauds we caught)')
plt.ylabel('Precision (How many flags were actual fraud)')
plt.title('Precision-Recall Curve  with SMOTE')
plt.legend()
plt.tight_layout()
plt.grid(True)
plt.show()

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_probs_smoteenn)
pr_auc = auc(recall, precision)

print(f"Area Under Precision-Recall with SMOTEENN: {pr_auc:.4f}")

plt.figure(figsize=(7, 5))
plt.plot(recall, precision, label=f"PR-AUC = {pr_auc:.3f}")
plt.xlabel('Recall (How many frauds we caught)')
plt.ylabel('Precision (How many flags were actual fraud)')
plt.title('Precision-Recall Curve with SMOTEENN and SMOTE')
plt.legend()
plt.tight_layout()
plt.grid(True)
plt.show()

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_probs_smoteenn_smote)
pr_auc = auc(recall, precision)

print(f"Area Under Precision-Recall with SMOTEENN and SMOTE: {pr_auc:.4f}")

plt.figure(figsize=(7, 5))
plt.plot(recall, precision, label=f"PR-AUC = {pr_auc:.3f}")
plt.xlabel('Recall (How many frauds we caught)')
plt.ylabel('Precision (How many flags were actual fraud)')
plt.title('Precision-Recall Curve with SMOTEENN and SMOTE')
plt.legend()
plt.tight_layout()
plt.grid(True)
plt.show()